> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验二：基于CANN的RoPE基础版算子实验


建议学时：4 学时


# 实验任务


## 任务描述


本实验围绕Qwen2.5的RoPE计算构建AscendC自定义算子，主要指导学生从kernel/host端Ascend代码开发开始，通过在PyTorch中注册直调核函数、验证单算子数学正确性，完成将自定义NPU算子嵌入Qwen2.5-0.5B模型中通过实机计时的完整流程。


## 学习目标


完成本任务的学习后，你应当能理解RoPE位置编码的数学语义与计算逻辑、基于Ascend C完成RoPE算子的设备端计算、完成PyTorch框架下直调核函数的注册与调用、设计单算子的正确性测试方法、通过手动撰写完整前向传播或采用Monkey Patch方法嵌入Qwen模型并实测性能、掌握基础算子的开发流程。


# 任务准备


## 算子定义


RoPE算子的数学定义为，对于词位置为第m个，词维度对数为第i对的局部向量和应当按进行旋转:


一般地，，其中d为总维度对数。


## RoPE简介


RoPE算子属于位置编码算子：在Attention模块中，计算Q、矩阵之间的矩阵乘法实质上是在批量计算词向量的q、k向量之间的点积所得权重，而不天然具备LSTM等循环神经网络中对先后顺序的感知能力。Attention的计算过程中，每个词向量转化所得的q、k向量必须手动嵌入词向量的位置信息以确保语序信息的正常传播。


假若Attention模块中缺少位置编码，语序信息将无法被注意力计算过程所感知。譬如：“死马当活马医”和“活马当死马医”这两句具有完全不同的含义的表达，由于缺少位置编码，将会计算出相同的输出：因为当两句话中词向量划分相同、每个词向量经过相同的线性变换，其q、k、v向量也就必然相同。


因此，需要加入位置编码，从而对令每对q、k在不同的位置下能分辨出不同的含义。RoPE的位置编码数学计算如上小节定义，本质为按照不同的位置，两两旋转一个词向量中的成对维度，相同的词语在不同的位置下，旋转的角度也不同。因此当两个词向量的q、k向量点乘时，将根据相对位置的不同，每个维度对产生不同程度的响应，从而感知语序信息。


## 算子定义与接口约定


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">input、cos、sin、output均为float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">二维连续Tensor 形状均为[totalTokens, headDim] cos/sin与input一一对应</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">totalTokens=batch×numHeads×seqLen 由调用方展平后传入</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum=min(8, totalTokens) rowsPerCore=(totalTokens+coreNum-1)/coreNum向上取整 每核处理连续的若干完整行，末核按rowEnd截断</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">totalTokens=128, headDim=64, base=1e6, coreNum=8</td>
</tr>
</tbody></table>


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0 需先source set_env.sh</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extention torch.ops.qwen_rope_custom.rope_baseline(Tensor x, Tensor cos, Tensor sin) -&gt; Tensor</td>
</tr>
<tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">FP32</td>
</tr>
<tr>
<td style="text-align:left;">构建结果</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、librope_torch_register.so；out/bin/rope_baseline_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">NumPy float64精度 覆盖三种接口的单算子正确性 + Qwen模型前向替换 + NPU-resident端到端（test_npu_e2e.py）</td>
</tr>
</tbody></table>


# 任务实施


## 步骤一：定义I/O规格与分块任务划分


按照算子数学公式定义，与输入张量形状，确定张量形状。根据RoPE的rotate_half公式明确输入为x、cos、sin三个二维张量，输出为y张量，形状均为 [totalTokens, headDim]，算子不改变张量形状。


约定张量内部数据类型、内存布局与算子调用接口签名。约定所有数据为FP32类型、连续内存布局，算子调用接口签名明确为接收三个Tensor返回一个Tensor。


设计数据传输结构体，由于基础版kernel全部采用GlobalTensor标量访问，仅需设计一个最小Tiling结构体传递totalTokens、headDim、coreNum和rowsPerCore四个运行时参数，复杂的分块策略留待优化版实现。


Host将运行时信息打包为16字节Tiling结构体。由于基础版Tiling全部为uint32_t整数字段，不存在浮点位模式被错误解释的风险，kernel侧可直接按uint32_t整体拷贝。Tiling定义为：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct RoPeTiling { uint32_t totalTokens = 0; // 总token行数 uint32_t headDim = 0; // head维度 uint32_t coreNum = 1; // 启动AI Core数量 uint32_t rowsPerCore = 0; // 每核处理行数 }; #pragma pack(pop)</th>
</tr>
</thead>
</table>


字段含义：totalTokens为展平后的总行数，headDim为每行元素数，coreNum为基础版固定8，rowsPerCore由 (totalTokens + coreNum - 1) / coreNum向上取整，末核通过rowEnd截断处理尾部。


## 步骤二：编写核心计算核函数


创建op_kernel目录，编写kernel头文件和入口文件。


头文件定义Kernel类，Init方法通过SetGlobalBuffer将GM_ADDR绑定为GlobalTensor句柄，同时接收Tiling参数并缓存到成员变量。


实现Process方法按core分工执行核心计算。Process方法首先通过GetBlockIdx计算当前Core的行区间，每个Core的行级分工逻辑如下，末核自动截断尾部：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">const uint32_t coreId = GetBlockIdx(); const uint32_t startRow = coreId * rowsPerCore_; uint32_t endRow = startRow + rowsPerCore_; if (endRow &gt; totalTokens_) endRow = totalTokens_;</th>
</tr>
</thead>
</table>


然后按每行headDim/2次迭代，逐元素从input、cos、sin中读取6个标量，按rotate_half公式计算两个结果并写回output。


核心计算摘录自op_kernel/rope_baseline_kernel.h，该路径刻意保留直接GM标量访问，用作功能和性能基线。每行headDim/2次迭代中执行6次标量读和2次标量写：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">const uint32_t halfHead = headDim_ / 2; for (uint32_t row = startRow; row &lt; endRow; row++) { const uint32_t rowOffset = row * headDim_; for (uint32_t i = 0; i &lt; halfHead; i++) { float x0 = inputGm_.GetValue(rowOffset + i); float x1 = inputGm_.GetValue(rowOffset + i + halfHead); float c0 = cosGm_.GetValue(rowOffset + i); float s0 = sinGm_.GetValue(rowOffset + i); float c1 = cosGm_.GetValue(rowOffset + i + halfHead); float s1 = sinGm_.GetValue(rowOffset + i + halfHead); outputGm_.SetValue(rowOffset + i, x0 * c0 - x1 * s0); outputGm_.SetValue(rowOffset + i + halfHead, x1 * c1 + x0 * s1); } }</th>
</tr>
</thead>
</table>


入口文件提供extern "C" global aicore修饰的kernel函数，同时可额外提供一个extern "C" 包装函数供Python ctypes直接调用，使得kernel同时支持ACLRT_LAUNCH_KERNEL宏和三尖括号启动两种方式。Kernel入口只负责从GM解包Tiling并创建实现对象，真正计算位于头文件。入口完整形式为：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">extern &quot;C&quot; global aicore void rope_baseline_kernel( GM_ADDR input, GM_ADDR cos_in, GM_ADDR sin_in, GM_ADDR output, GM_ADDR workspace, GM_ADDR tiling) { RoPeTiling t; const gm uint32_t *src = reinterpret_cast&lt;const gm uint32_t *&gt;(tiling); uint32_t *dst = reinterpret_cast&lt;uint32_t *&gt;(&amp;t); for (uint32_t i = 0; i &lt; sizeof(RoPeTiling) / sizeof(uint32_t); ++i) dst[i] = src[i]; KernelRoPeBaseline op; op.Init(input, cos_in, sin_in, output, t.totalTokens, t.headDim, t.coreNum, t.rowsPerCore); op.Process(); }</th>
</tr>
</thead>
</table>


## 步骤三：编写主机侧的调用与算子注册


Host端工作分为两个独立目标。


其一是独立可执行验证程序，位于verify_kernel_launch目录，完成完整的ACL生命周期：aclInit初始化设备、aclrtMalloc分配设备端输入/输出/Tiling缓冲区、aclrtMemcpy执行H2D拷贝、ACLRT_LAUNCH_KERNEL启动核函数、aclrtSynchronizeStream同步、最后aclrtMemcpy执行D2H回读——整套流程不依赖PyTorch，适合作为最小可复现单元，可作为算子最基本的编译测试。


其二是torch_extension注册文件，通过TORCH_LIBRARY宏将算子注册到torch.ops命名空间，TORCH_LIBRARY_IMPL绑定C++ 实现函数。实现函数内部封装同样的ACL生命周期，并引入静态全局设备缓存，避免每次调用重新分配和释放设备内存。注册文件在调用ACL前对输入进行防御性校验，确保输入为CPU上的float32张量且cos/sin形状与x匹配：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">auto x_c = x.contiguous().to(at::kFloat); auto cos_c = cos.contiguous().to(at::kFloat); auto sin_c = sin.contiguous().to(at::kFloat); TORCH_LIBRARY(qwen_rope_custom, m) { m.def(&quot;rope_baseline(Tensor x, Tensor cos, Tensor sin) -&gt; Tensor&quot;); } TORCH_LIBRARY_IMPL(qwen_rope_custom, CompositeExplicitAutograd, m) { m.impl(&quot;rope_baseline&quot;, rope_baseline_npu); }</th>
</tr>
</thead>
</table>


此外编写init.py，在import时自动检测CANN环境、注入LD_LIBRARY_PATH、加载编译好的 .so文件，使Python侧只需from torch_extension import load_torch_ops即可完成全部初始化。


## 步骤四：Cmake目标构建


构建系统需要同时产出两套性质不同的二进制产物。


通过CMake的ExternalProject机制，先由ascendc_library编译设备端异构程序，经毕昇编译器生成AI Core目标码，打包为libascendc_kernels_npu.so并自动生成aclrtlaunch_rope_baseline_kernel.h启动头文件。前者为NPU运行的实际代码，后者为向Host侧提供核函数的注册以及启动方式。


随后用宿主g++编译器分别编译独立可执行测试文件和torch注册动态库librope_torch_register.so，前者为调用aclrtlaunch_rope_basline_kernel.h的独立可执行文件，后者为PyTorch的算子注册动态库，该库同时链接libtorch_cpu.so、libascendcl.so和前述libascendc_kernel_npu.so，且在内部封装了NPU完整的启动、设备分配、内存管理、同步与设备端缓存。


通过配置include路径和RPATH，或借助run_test.sh注入LD_LIBRARY_PATH，保证两套产物均能在运行时正确解析所有依赖，共存于统一的out/lib和out/bin目录下。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">张量</th>
<th style="text-align:left;">最大误差</th>
<th style="text-align:left;">平均误差</th>
<th style="text-align:left;">判定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">[128,64]</td>
<td style="text-align:left;">2.38418579e-07</td>
<td style="text-align:left;">1.01561461e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
<tr>
<td style="text-align:left;">[128,128]</td>
<td style="text-align:left;">2.38418579e-07</td>
<td style="text-align:left;">1.02753424e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
<tr>
<td style="text-align:left;">[256,64]</td>
<td style="text-align:left;">2.38418579e-07</td>
<td style="text-align:left;">1.10366223e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
<tr>
<td style="text-align:left;">[64,128]</td>
<td style="text-align:left;">2.38418579e-07</td>
<td style="text-align:left;">1.10366223e-08</td>
<td style="text-align:left;">PASS</td>
</tr>
</tbody></table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /YOURPATH/RopeBaselineExperiment bash scripts/check_env.sh bash scripts/build.sh bash scripts/run_test.sh tests/test_torch_op.py # 单算子正确性 bash scripts/run_test.sh tests/test_qwen_forward.py # Qwen链路替换验证 bash scripts/run_test.sh tests/test_mini_qwen.py # 最小Qwen模型端到端 TOTAL_TOKENS=*** HEAD_DIM=64 BLOCK_DIM=8 WARMUP=5 REPEAT=10 bash scripts/profile.sh</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RopeBaselineExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/RopeBaselineExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
mkdir -p input output
python3 -c "import numpy as np; t,d=128,64; r=np.random.default_rng(42); x=r.normal(size=(t,d)).astype(np.float32); f=np.outer(np.arange(t,dtype=np.float64),1.0/(1000000.0**(np.arange(0,d,2,dtype=np.float64)/d))); e=np.concatenate([f,f],axis=-1).astype(np.float32); x.tofile('input/input_x.bin'); np.cos(e).tofile('input/input_cos.bin'); np.sin(e).tofile('input/input_sin.bin')"
./out/bin/rope_baseline_standalone --tokens 128 --head-dim 64 --block-dim 8 --warmup 20 --repeat 100 --rounds 5


# 任务拓展


## 参数扫描与性能分析


改变tokens、headDim和blockDim，各自独立运行独立测试可执行文件。观察：短序列下核调度开销是否超过计算时间，表现为blockDim增大而device时间不降反升。


## msprof瓶颈定位


在典型形状下运行msprof采集AI Core指标。重点关注：


aiv_scalar_time与aiv_vec_time的比值，由于基础版采用标量读逐个取值计算，预期scalar_time占绝对主导，向量单元几乎空闲。


aic_mte_time，即DMA时间，验证逐元素读写是否造成MTE流水线空转。


对比HBM标称带宽1200GB/s，基础版预期有效带宽远低于峰值，确认瓶颈在访问模式而非算力。


# 实验总结


本实验完成了RoPE基础版算子的完整开发闭环：从数学公式出发，定义输入输出规格与Tiling数据，编写AscendC kernel实现rotate_half的逐元素计算，构建Host端独立验证程序和torch.library注册动态库，NumPy float64 golden验证四组shape全部在1e-3阈值内通过，后使用ACL Event计时获得22.5 us的device侧kernel性能基线。


基础版的核心价值在于基础功能的实现，每行headDim/2次迭代中的6次GM读和2次GM写直接对应rotate_half公式的展开，无缓冲复用、无流水线重叠、无数据重排。学生通过阅读kernel代码即可逐行对账数学语义，这是后续所有优化版本的正确性锚点。


本实验建立了算子开发的标准流程：


定义规格 → 编写kernel → 注册调用 → 构建验证 → Golden测试	 → 计时分析。
